In [18]:
import transformer_lens as tl
from transformer_lens import HookedTransformer, HookedTransformerConfig
import torch
import pandas as pd
import numpy as np
from lens import tuned_lens, logit_lens

Change device to cpu if no cuda device is available.

In [ ]:
device = 'cuda:0'
batch_size = 50
context_window = 10
num_text_samples = 10 # Decrement this number if you need to reduce training time for testing purposes
num_batches = num_text_samples // batch_size
print(f'Expected number of iterations: {num_batches}')

In [20]:
from datasets import load_dataset
model: HookedTransformer = tl.HookedTransformer.from_pretrained("pythia-14m",device=device)
num_model_blocks = len(model.blocks)
# Entries 0..num_model_blocks-1 are block probes; the last entry is the final output head.

tl = tuned_lens(model, device)
ll = logit_lens()

unembedding_matrices = tl.unembedding_matrices

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 18167.51it/s]


Loaded pretrained model pythia-14m into HookedTransformer


### Load and clean pile-10k dataset, initialize data loader

In [21]:
ds = load_dataset("NeelNanda/pile-10k")
cleaned_text = ds['train']['text'][0]
tokens = torch.zeros(1, context_window+1).to(device)
torch.manual_seed(0)
rng = np.random.default_rng(seed=0)
randoms = torch.from_numpy(rng.choice(len(ds['train']), size=num_text_samples, replace=False)).to(device)
for i in range(num_text_samples):
    texty_text = ds['train']['text'][randoms[i].item()]
    encoded = model.tokenizer.encode(texty_text)
    if len(encoded) < context_window+1:
        continue
    tokens_add = torch.tensor(encoded).unfold(0, context_window+1, 1).to(device)
    tokens = torch.cat((tokens, tokens_add), dim=0)
tokens = tokens[1:].long()
data_loader = torch.utils.data.DataLoader(tokens, batch_size=batch_size, shuffle=True)


### Method to save tuned lens and training stats.

In [22]:
from datetime import datetime
from pathlib import Path
import pandas as pd

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
save_base_model_state = False

def save_training_results(run_name, tl: tuned_lens, training_history=None, extra=None, save_base_model_state=False):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    lens_path = results_dir / f"{run_name}_{timestamp}.pt"
    history_path = results_dir / f"{run_name}_{timestamp}_training.csv"

    # Save only the tuned lens weights with the tuned_lens built-in method.
    tl.save_lens(lens_path)

    metadata = {
        "run_name": run_name,
        "lens_path": str(lens_path),
        "model_name": getattr(model.cfg, "model_name", None),
        "num_model_blocks": num_model_blocks,
        "context_window": context_window,
        "num_text_samples": num_text_samples,
        "batch_size": batch_size,
        "num_epochs": num_epochs,
    }
    if extra is not None:
        metadata.update(extra)

    rows = []
    for entry in training_history or []:
        row = {**metadata}
        for key, value in entry.items():
            if key != "block_losses":
                row[key] = value
        for layer, loss in enumerate(entry.get("block_losses", [])):
            row[f"block_{layer}_loss"] = loss
        rows.append(row)

    pd.DataFrame(rows or [metadata]).to_csv(history_path, index=False)
    print(f"Saved tuned lens to {lens_path}")
    print(f"Saved training history to {history_path}")
    return {"lens_path": lens_path, "history_path": history_path}

num_epochs =1
criterion = torch.nn.CrossEntropyLoss()
print(criterion(torch.tensor([[0.1, 0.2, 0.3], [0, 0, 0]]), torch.tensor([2, 1])))
optimizer_list = [torch.optim.Adam(head.parameters(), lr=1e-3) for head in unembedding_matrices]

tensor(1.0503)


## Train the tuned lens

In [23]:
# Freeze the base model; only train the probe heads and final output head in this loop.
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
unembedding_matrices.train()
probe_training_history = []

for epoch in range(num_epochs):
    iteration = 0
    for batch in data_loader:
        # print(batch)
        with torch.no_grad():
            _, cache = model.run_with_cache(batch[:, :-1])
        targets = batch[:, 1:]
        block_loss_values = []
        for i in range(num_model_blocks):
            optimizer_list[i].zero_grad()
            # CHANGED: detach to avoid backprop into the model when training only unembedding matrices
            block_output = cache["resid_post", i]
            # Train without applying the final layer norm.
            output = unembedding_matrices[i](block_output)
            # CHANGED: CrossEntropyLoss expects class indices, not one-hot targets
            loss = criterion(output.reshape(-1, output.shape[-1]), targets.reshape(-1))
            loss.backward()
            optimizer_list[i].step()
            block_loss_values.append(loss.item())
        probe_training_history.append({
            "epoch": epoch,
            "iteration": iteration,
            "block_losses": block_loss_values,
        })

        if iteration % 100 == 0:
            print(
                f"Epoch {epoch}, Iteration {iteration}, {loss.item():.4f}, Block losses: {[f'{bl:.4f}' for bl in block_loss_values]}"
            )
        iteration += 1




Epoch 0, Iteration 0, 10.8290, Block losses: ['10.8262', '10.8307', '10.8616', '10.8698', '10.8685', '10.8290']
Epoch 0, Iteration 100, 9.6245, Block losses: ['10.5025', '10.4301', '10.1246', '9.9880', '9.7275', '9.6245']
Epoch 0, Iteration 200, 8.7912, Block losses: ['10.1873', '10.0914', '9.8102', '9.5136', '9.0984', '8.7912']
Epoch 0, Iteration 300, 8.1550, Block losses: ['9.9354', '9.7802', '9.4801', '9.0775', '8.5722', '8.1550']
Epoch 0, Iteration 400, 7.9701, Block losses: ['9.7214', '9.5304', '9.3410', '8.7942', '8.3148', '7.9701']
Epoch 0, Iteration 500, 7.8681, Block losses: ['9.5230', '9.3294', '9.1960', '8.6191', '8.0970', '7.8681']
Epoch 0, Iteration 600, 7.4277, Block losses: ['9.2656', '9.0452', '8.8591', '8.1914', '7.6446', '7.4277']
Epoch 0, Iteration 700, 7.3816, Block losses: ['9.1062', '8.9048', '8.8212', '8.0725', '7.6474', '7.3816']
Epoch 0, Iteration 800, 7.5916, Block losses: ['9.0015', '8.8422', '8.7732', '8.0509', '7.7361', '7.5916']
Epoch 0, Iteration 900, 6.9

In [24]:
probe_results_path = save_training_results(
    "probe_heads",
    tl,
    training_history=probe_training_history,
    extra={"probe_input_normalization": "raw_resid_post"},
    save_base_model_state=False,
)

Saved tuned lens to results/probe_heads_20260509_005342.pt
Saved training history to results/probe_heads_20260509_005342_training.csv
